In [ ]:
import librosa
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

# 1. Define model and load components
# Use "google/gemma-4-E2B-it" for the instruction-tuned version
model_id = "google/gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
)

# 2. Load and preprocess the audio file
# Gemma 4 expects 16,000Hz sampling rate mono audio (Max length: 30 seconds)
audio_path = "../../.data/persistent/Ta_c_ang_s_ng_b_ng_nh_ng_l_i_khen_-_Meichan_Kh_ng_Th_ng_SS3__turn027_spk_02_83.63-85.72.wav"
audio_array, sampling_rate = librosa.load(audio_path, sr=16000, mono=True)

# 3. Create the chat structure with the special <audio> token
messages = [
    {
        "role": "user",
        "content": [
            {"type": "audio"},
            {
                "type": "text",
                "text": "Transcribe the following speech segment in English into English text.",
            },
        ],
    }
]

# 4. Format prompt through the processor
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(
    text=prompt,
    audios=audio_array,
    sampling_rate=sampling_rate,
    return_tensors="pt",
)

# Move inputs to the same device as the model
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# 5. Generate response
# For ASR (Speech-to-Text), Google recommends do_sample=False
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

# Decode and print the transcription/answer
# Slice output_ids to exclude the prompt tokens if necessary
generated_ids = output_ids[:, inputs["input_ids"].shape[1] :]
response = processor.batch_decode(
    generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print("Model Output:", response[0])
